In [ ]:

!pip -q install python-docx pdfplumber openpyxl

import re
import json
import os
import pandas as pd
import unicodedata
from difflib import SequenceMatcher

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.0/71.0 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 44.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 65.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 47.1 MB/s eta 0:00:00


In [23]:

# ============================================================
# COLAB PACKAGE EXTRACTOR - JSON ONLY OUTPUT
# Fixed for packages that follow this structure:
# Trip Overview -> The Invitation -> Experiences -> The Journey
# -> What's Included -> What You'll Need to Arrange -> Equipment
# -> Safety & Guide Judgment -> Physical Preparation -> Common Curiosities
# ============================================================

try:
    from google.colab import files
except Exception:
    files = None


import os
import re
import json
from difflib import SequenceMatcher

# ------------------------------------------------------------
# 1. OPTIONAL MANUAL INPUTS
# Fill only when known. Empty values are kept as None or omitted.
# ------------------------------------------------------------

MANUAL_INPUTS = {
    # Leave empty to auto-generate from seasons.
    # ISO example: ["2026-09-01T00:00:00.000Z"]
    "departureDates": [],
    "isActive": True,              # True / False
    "pricePerPerson": "0",         # example: "1499"
    "paymentPolicy": "",
    "budget": "midrange",         # budget / midrange / luxury
    "tier": "standard",           # starter / standard / premium
    "badge": "listed",            # listed / promising / reliable / top_rated / iconic
    "currency": "USD",            # example: "USD" or "NPR".
}

# ------------------------------------------------------------
# 2. VALID SCHEMA ENUMS
# These are fixed schema slugs/enums. Output must use these exact values only.
# ------------------------------------------------------------

VALID_AGENCIES = ['Amigo Trekking',
 'Caravan Outdoors',
 'Explore K2 Pakistan',
 'Go The Himalaya Treks',
 'Himalayan Gateway Trek',
 'Himalayan Namobuddha Travel & Treks',
 'Info Nepal Treks and Expedition',
 'Karakoram Calling Treks',
 'Magic Expedition & Tours',
 'Moon Himilayan Adventure Pvt Ltd',
 'Nepal Hiking Adventure',
 'Nepal Royal Tourism Holiday',
 'Nepal Unique Treks',
 'Pokhara Mountain Bike Adventure',
 'Tanzania Agency']

VALID_SUBCATEGORIES = ["lakes-hill-stations",
                       "heli-treks-heli-returns",
                       "secluded-boutique-heritage",
                       "rural-heritage-homestays",
                       "city-day-tours",
                       "peak-climbing-tiers",
                       "aerial-sky",
                       "ayurveda-detox-panchakarma",
                       "meditation-vipassana",
                       "easy-treks-nature-walks",
                       "voluntourism-conservation-stays",
                       "yoga-retreats",
                       "expeditions-7000-8000m",
                       "mindful-trekking-nature-immersion",
                       "land-vertical",
                       "national-parks-safaris",
                       "weekend-short-breaks",
                       "unesco-ancient-capitals",
                       "bungee-extreme",
                       "water-canyon",
                       "bike-moto",
                       "spa-resorts-hot-springs",
                       "mountain-sunrise-escapes",
                       "pilgrimage-circuits",
                       "extensions-multi-country",
                       "classic-short-treks",
                       "adventure-for-two",
                       "remote-restricted-area-treks",
                       "tailor-made-vip-concierge",
                       "lakeside-romance-pokhara",
                       "interactive-learning",
                       "luxury-spa-stays",
                       "cycling-mtb-endurance",
                       "festivals-ethnic-calendars",
                       "photography-filming-permits"
]

VALID_TRIPS = ["langtang-ganja-la-pass-trek",
"kanchenjunga-south-base-camp-trek",
"skardu-valley",
"barun-valley-trek-packages-route-and-planner",
"k2-base-camp-trek-packages-route-and-planner",
"zanzibar-island-experience-packages-route-and-planner",
"rara-short-trek-packages-route-and-planner",
"muldai-view-point-trek",
"island-peak-with-ebc-trek-packages-route-and-planner",
"khopra-ridge-trek-packages-route-and-planner",
"ice-lake-trek-manang-packages-route-and-planner",
"dolpo-shey-festival-trek",
"khopra-ridge-trek",
"dhorpatan-dolpo-trek-packages-route-and-planner",
"gauri-shankar-himal-trekking",
"phoksundo-lake-trek-packages-route-and-planner",
"upper-mustang-trek-packages-route-and-planner",
"mountain-bike-in-pokhara-packages-route-and-planner",
"short-manaslu-circuit-trek-packages-route-and-planner",
"upper-dolpo-trekking-packages-route-and-planner",
"api-himal-base-camp-trek-packages-route-and-planner",
"tiji-festival-trek",
"gondogoro-la-pass-trek-packages-route-and-planner",
"short-makalu-base-camp-trek-packages-route-and-planner",
"annapurna-circuit-trekking",
"short-langtang-valley-trek",
"jumla-to-rara-lake-trek-packages-route-and-planner",
"rara-lake-trek-packages-route-and-planner",
"mardi-himal-trek-packages-route-and-planner",
"gosaikunda-lake-trek",
"short-everest-base-camp-trek-packages-route-and-planner",
"budget-everest-base-camp-trek-packages-route-and-planner",
"rara-lake-yoga-trek-packages-route-and-planner",
"dhaulagiri-sanctuary-trek-packages-route-and-planner",
"bhaktapur-tour-packages-route-and-planner",
"mera-peak-climbing-packages-route-and-planner",
"islamabad-city-tour-packages-route-and-planner",
"laila-peak-base-camp-trek-packages-route-and-planner",
"limi-valley-trek-packages-route-and-planner",
"dhaulagiri-circuit-trek-packages-route-and-planner",
"annapurna-sanctuary-trek-packages-route-and-planner",
"rara-lake-and-khaptad-circuit-trek-packages-route-and-planner",
"makalu-base-camp-trek-packages-route-and-planner",
"yalung-ri-peak-climbing-packages-route-and-planner",
"kanchenjunga-north-base-camp-trek",
"gokyo-valley-trek-packages-route-and-planner",
"annapurna-circuit-mountain-bike-tour-packages-route-and-planner",
"spantik-peak-expedition-packages-route-and-planner",
"upper-mustang-bike-tour-packages-route-and-planner",
"tilicho-lake-trek",
"everest-three-passes-trek-packages-route-and-planner",
"annapurna-panorama-trek",
"ghorepani-poon-hill-trek",

"dhaulagiri-base-camp-trek-packages-route-and-planner",
"kanchenjunga-base-camp-trek",
"k7-base-camp-trek-packages-route-and-planner",
"dhorpatan-trek-packages-route-and-planner",
"pachermo-peak-climbing-packages-route-and-planner",
"hunza-valley-tours",
"dhampus-peak-climbing-packages-route-and-planner",
"kanchenjunga-circuit-trek",
"everest-base-camp-trek-with-helicopter-return-packages-route-and-planner",
"bigu-gompa-trek",
"tanzania-packages-route-and-planner",
"tsho-rolpa-trek-packages-route-and-planner",
"khaptad-national-park",
"chisapani-nagarkot-trekking",
"ama-yangri-trek",
"nar-phu-valley-trek",
"dhaulagiri-plus-upper-mustang-trek-packages-route-and-planner",
"nar-phu-to-upper-mustang-trek-packages-route-and-planner",
"guerrilla-trek-packages-route-and-planner",
"gasherbrum-massif",
"gokyo-lakes-trek",
"mohare-danda-trek",
"namche-bazaar-trek",
"rolwaling-valley-trek",
"ghandruk-trek",
"machhapuchhre-model-trek-packages-route-and-planner",
"tamang-heritage-trek",
"tsum-valley-trek-packages-route-and-planner",
"panch-pokhari-trek",
"helambu-trek",
"sikles-kapuche-lake-trek-packages-route-and-planner",
"dolpo-mustang-traverse-trek-packages-route-and-planner",
"lower-dolpo-trekking-packages-route-and-planner",
"everest-panorama-trek-packages-route-and-planner",
"pastore-base-camp-trek-packages-route-and-planner",
"lower-mustang-trek",
"island-peak-climbing-packages-route-and-planner",
"langtang-valley-trek",
"everest-base-camp-heli-trek-packages-route-and-planner",
"jiri-to-everest-base-camp-trek-packages-route-and-planner",
"jomsom-muktinath-trek",
"tashi-lapcha-pass-trek-packages-route-and-planner",
"everest-base-camp-trek-by-road-packages-route-and-planner",
"kathmandu-valley-tour-heritage-temples-and-living-culture",
"arun-valley-trek-packages-route-and-planner",
"mt-nanga-parbat-expedition",
"dolpo-rara-traverse-trek-packages-route-and-planner",
"mount-k2-expedition",
"mesokanto-la-pass-trek-packages-route-and-planner",
"ama-dablam-base-camp-trek-packages-route-and-planner",
"luxury-everest-base-camp-trek-packages-route-and-planner",
"pikey-peak-trek-packages-route-and-planner",
"annapurna-base-camp-trek-packages-route-and-planner",
"patan-tour-packages-route-and-planner",
"lower-mustang-mountain-biking-packages-route-and-planner",
"manaslu-circuit-trek"]

VALID_DESTINATIONS = ["pokhara",
"langtang",
"rara-lake",
"annapurna-region",
"everest-region",
"chitwan-national-part",
"bandipur",
"bardia-national-park",
"upper-mustang",
"janakpur",
"nagarkot",
"kathmandu",
"patan",
                      "bhaktapur","lumbini","dhaulagiri-region","far-west-region","makalu-barun-region","kanchenjunga-region",
                      "rolwaling-region","dolpo-region","manaslu-region","langtang-region"
]

VALID_SEASONS = ['spring', 'summer', 'autumn', 'winter']

VALID_DIFFICULTIES = ['beginner', 'moderate', 'difficult', 'extremely_difficult']

VALID_BUDGETS = ['budget', 'midrange', 'luxury']

VALID_TIERS = ['starter', 'standard', 'premium']

VALID_BADGES = ['listed', 'promising', 'reliable', 'top_rated', 'iconic']


# ------------------------------------------------------------
# 2B. SCHEMA VALUE HELPERS
# These helpers convert older display labels into schema slugs.
# ------------------------------------------------------------

SUBCATEGORY_ALIASES = {
    "City Day Tours & Sightseeing": "city-day-tours",
    "City Day Tours": "city-day-tours",
    "Expeditions (7000-8000m)": "expeditions-7000-8000m",
    "Expeditions 7000-8000m": "expeditions-7000-8000m",
    "Bike & Moto": "bike-moto",
    "Heli Treks & Heli Returns": "heli-treks-heli-returns",
    "Aerial & Sky": "aerial-sky",
    "Yoga Retreats": "yoga-retreats",
    "Photography, Filming & Permits": "photography-filming-permits",
    "Lakes & Hill Stations": "lakes-hill-stations",
    "Weekend & Short Breaks": "weekend-short-breaks",
    "Easy Treks & Nature Walks": "easy-treks-nature-walks",
    "Mountain & Sunrise Escapes": "mountain-sunrise-escapes",
    "Classic & Short Treks": "classic-short-treks",
    "Mindful Trekking & Nature Immersion": "mindful-trekking-nature-immersion",
    "Land & Vertical": "land-vertical",
    "Water & Canyon": "water-canyon",
    "Extensions & Multi-Country": "extensions-multi-country",
    "Adventure for Two": "adventure-for-two",
    "Lakeside Romance (Pokhara)": "lakeside-romance-pokhara",
    "Meditation & Vipassana": "meditation-vipassana",
    "Ayurveda & Detox (Panchakarma)": "ayurveda-detox-panchakarma",
    "National Parks & Safaris": "national-parks-safaris",
    "Remote & Restricted Area Treks": "remote-restricted-area-treks",
    "Peak Climbing Tiers": "peak-climbing-tiers",
}

DESTINATION_ALIASES = {
    "Pokhara": "pokhara",
    "Langtang": "langtang",
    "Rara Lake": "rara-lake",
    "Annapurna Region": "annapurna-region",
    "Everest Region": "everest-region",
    "Chitwan National Park": "chitwan-national-part",
    "Bandipur": "bandipur",
    "Bardia National Park": "bardia-national-park",
    "Upper Mustang": "upper-mustang",
    "Janakpur": "janakpur",
    "Nagarkot": "nagarkot",
    "Kathmandu": "kathmandu",
    "Patan": "patan",
    "Bhaktapur": "bhaktapur",
    "Lumbini": "lumbini",
    "Dhaulagiri region": "dhaulagiri-region",
    "Dhaulagiri Region": "dhaulagiri-region",
    "Far West Region": "far-west-region",
    "Makalu Barun region": "makalu-barun-region",
    "Makalu Barun Region": "makalu-barun-region",
    "Kanchenjunga region": "kanchenjunga-region",
    "Kanchenjunga Region": "kanchenjunga-region",
    "Rolwaling Region": "rolwaling-region",
    "Dolpo Region": "dolpo-region",
    "Manaslu Region": "manaslu-region",
    "Langtang Region": "langtang-region",
}

TRIP_ALIASES = {
    # Direct simple aliases. These are used only after the higher-priority
    # route-specific rules below.
    "poon hill": "ghorepani-poon-hill-trek",
    "ghorepani": "ghorepani-poon-hill-trek",
    "annapurna base camp": "annapurna-base-camp-trek-packages-route-and-planner",
    "abc trek": "annapurna-base-camp-trek-packages-route-and-planner",
    "annapurna sanctuary": "annapurna-sanctuary-trek-packages-route-and-planner",
    "annapurna circuit": "annapurna-circuit-trekking",
    "k2 base camp": "k2-base-camp-trek-packages-route-and-planner",
    "mount k2": "mount-k2-expedition",
    "nanga parbat": "mt-nanga-parbat-expedition",
}


# More specific rules must stay BEFORE generic/fuzzy matching.
# Each entry is:
# ("valid-trip-slug", [["all", "phrases", "must", "match"], ["alternative", "group"]])
TRIP_PRIORITY_RULES = [
    # Everest / Khumbu variants
    ("island-peak-with-ebc-trek-packages-route-and-planner", [["island peak", "everest base camp"], ["island peak", "ebc"]]),
    ("island-peak-climbing-packages-route-and-planner", [["island peak climbing"], ["island peak"]]),
    ("everest-base-camp-trek-with-helicopter-return-packages-route-and-planner", [["everest base camp", "helicopter return"], ["ebc", "heli return"], ["everest base camp", "heli return"]]),
    ("everest-base-camp-heli-trek-packages-route-and-planner", [["everest base camp", "heli trek"], ["ebc", "heli trek"], ["everest base camp", "helicopter trek"]]),
    ("everest-base-camp-trek-by-road-packages-route-and-planner", [["everest base camp", "by road"], ["ebc", "by road"], ["everest base camp", "drive"]]),
    ("jiri-to-everest-base-camp-trek-packages-route-and-planner", [["jiri", "everest base camp"], ["jiri", "ebc"]]),
    ("luxury-everest-base-camp-trek-packages-route-and-planner", [["luxury", "everest base camp"], ["luxury", "ebc"]]),
    ("budget-everest-base-camp-trek-packages-route-and-planner", [["budget", "everest base camp"], ["budget", "ebc"]]),
    ("short-everest-base-camp-trek-packages-route-and-planner", [["short", "everest base camp"], ["short", "ebc"], ["beginner", "everest base camp"], ["beginners", "everest base camp"], ["beginner", "ebc"], ["beginners", "ebc"]]),

    ("ama-dablam-base-camp-trek-packages-route-and-planner", [["ama dablam base camp"], ["ama dablam", "side trip"], ["ama dablam", "everest base camp"]]),
    ("gokyo-lakes-trek", [["gokyo lakes"], ["gokyo lake"], ["gokyo", "cho la"], ["gokyo", "everest base camp"]]),
    ("gokyo-valley-trek-packages-route-and-planner", [["gokyo valley"]]),
    ("everest-three-passes-trek-packages-route-and-planner", [["three passes"], ["kongma la"], ["cho la", "renjo la"], ["kongma", "renjo"], ["kongma", "cho la"]]),
    ("everest-panorama-trek-packages-route-and-planner", [["everest panorama"]]),
    ("namche-bazaar-trek", [["namche bazaar trek"], ["namche bazaar"]]),
    ("pikey-peak-trek-packages-route-and-planner", [["pikey peak"]]),

    # Annapurna variants
    ("annapurna-circuit-mountain-bike-tour-packages-route-and-planner", [["annapurna circuit", "mountain bike"], ["annapurna circuit", "mtb"], ["annapurna circuit", "biking"]]),
    ("annapurna-circuit-trekking", [["annapurna circuit"]]),
    ("annapurna-base-camp-trek-packages-route-and-planner", [["annapurna base camp"], ["abc trek"]]),
    ("annapurna-sanctuary-trek-packages-route-and-planner", [["annapurna sanctuary"]]),
    ("mardi-himal-trek-packages-route-and-planner", [["mardi himal"]]),
    ("ghorepani-poon-hill-trek", [["poon hill"], ["ghorepani"]]),
    ("khopra-ridge-trek-packages-route-and-planner", [["khopra ridge"]]),
    ("khopra-ridge-trek", [["khopra"]]),
    ("muldai-view-point-trek", [["muldai view point"], ["muldai viewpoint"], ["muldai"]]),
    ("mohare-danda-trek", [["mohare danda"]]),
    ("tilicho-lake-trek", [["tilicho lake"]]),
    ("ice-lake-trek-manang-packages-route-and-planner", [["ice lake", "manang"]]),
    ("jomsom-muktinath-trek", [["jomsom", "muktinath"]]),

    # Langtang / Rolwaling / Manaslu
    ("langtang-ganja-la-pass-trek", [["ganja la"], ["langtang", "ganja"]]),
    ("short-langtang-valley-trek", [["short", "langtang valley"]]),
    ("langtang-valley-trek", [["langtang valley"]]),
    ("gosaikunda-lake-trek", [["gosaikunda"]]),
    ("helambu-trek", [["helambu"]]),
    ("tamang-heritage-trek", [["tamang heritage"]]),
    ("tashi-lapcha-pass-trek-packages-route-and-planner", [["tashi lapcha"]]),
    ("pachermo-peak-climbing-packages-route-and-planner", [["pachermo"]]),
    ("yalung-ri-peak-climbing-packages-route-and-planner", [["yalung ri"]]),
    ("rolwaling-valley-trek", [["rolwaling"]]),
    ("short-manaslu-circuit-trek-packages-route-and-planner", [["short", "manaslu circuit"]]),
    ("manaslu-circuit-trek", [["manaslu circuit"], ["larkya la"]]),
    ("tsum-valley-trek-packages-route-and-planner", [["tsum valley"]]),

    # Mustang / Dolpo / Rara / Far West
    ("upper-mustang-bike-tour-packages-route-and-planner", [["upper mustang", "bike"], ["upper mustang", "biking"], ["upper mustang", "mtb"]]),
    ("upper-mustang-trek-packages-route-and-planner", [["upper mustang"], ["lo manthang"]]),
    ("lower-mustang-mountain-biking-packages-route-and-planner", [["lower mustang", "mountain biking"], ["lower mustang", "bike"], ["lower mustang", "mtb"]]),
    ("lower-mustang-trek", [["lower mustang"]]),
    ("tiji-festival-trek", [["tiji festival"], ["tiji"]]),
    ("phoksundo-lake-trek-packages-route-and-planner", [["phoksundo"]]),
    ("upper-dolpo-trekking-packages-route-and-planner", [["upper dolpo"]]),
    ("lower-dolpo-trekking-packages-route-and-planner", [["lower dolpo"]]),
    ("dolpo-shey-festival-trek", [["shey festival"], ["dolpo shey"]]),
    ("dolpo-mustang-traverse-trek-packages-route-and-planner", [["dolpo", "mustang", "traverse"]]),
    ("dolpo-rara-traverse-trek-packages-route-and-planner", [["dolpo", "rara", "traverse"]]),
    ("rara-lake-and-khaptad-circuit-trek-packages-route-and-planner", [["rara", "khaptad"]]),
    ("rara-lake-yoga-trek-packages-route-and-planner", [["rara", "yoga"]]),
    ("rara-short-trek-packages-route-and-planner", [["rara", "short"]]),
    ("jumla-to-rara-lake-trek-packages-route-and-planner", [["jumla", "rara"]]),
    ("rara-lake-trek-packages-route-and-planner", [["rara lake"], ["rara"]]),
    ("khaptad-national-park", [["khaptad"]]),
    ("api-himal-base-camp-trek-packages-route-and-planner", [["api himal"]]),

    # Kanchenjunga / Makalu / Dhaulagiri
    ("kanchenjunga-north-base-camp-trek", [["kanchenjunga north base camp"], ["north base camp", "kanchenjunga"]]),
    ("kanchenjunga-south-base-camp-trek", [["kanchenjunga south base camp"], ["south base camp", "kanchenjunga"]]),
    ("kanchenjunga-circuit-trek", [["kanchenjunga circuit"]]),
    ("kanchenjunga-base-camp-trek", [["kanchenjunga base camp"], ["kanchenjunga"]]),
    ("short-makalu-base-camp-trek-packages-route-and-planner", [["short", "makalu base camp"]]),
    ("makalu-base-camp-trek-packages-route-and-planner", [["makalu base camp"]]),
    ("barun-valley-trek-packages-route-and-planner", [["barun valley"]]),
    ("dhaulagiri-plus-upper-mustang-trek-packages-route-and-planner", [["dhaulagiri", "upper mustang"]]),
    ("dhaulagiri-sanctuary-trek-packages-route-and-planner", [["dhaulagiri sanctuary"]]),
    ("dhaulagiri-circuit-trek-packages-route-and-planner", [["dhaulagiri circuit"]]),
    ("dhaulagiri-base-camp-trek-packages-route-and-planner", [["dhaulagiri base camp"]]),

    # Pakistan / Karakoram / Tanzania / city tours
    ("gondogoro-la-pass-trek-packages-route-and-planner", [["gondogoro la"], ["gondogoro"]]),
    ("k2-base-camp-trek-packages-route-and-planner", [["k2 base camp"], ["baltoro", "concordia"]]),
    ("mount-k2-expedition", [["mount k2"], ["k2 expedition"]]),
    ("mt-nanga-parbat-expedition", [["nanga parbat"]]),
    ("spantik-peak-expedition-packages-route-and-planner", [["spantik"]]),
    ("laila-peak-base-camp-trek-packages-route-and-planner", [["laila peak"]]),
    ("pastore-base-camp-trek-packages-route-and-planner", [["pastore"]]),
    ("k7-base-camp-trek-packages-route-and-planner", [["k7 base camp"], ["k7"]]),
    ("gasherbrum-massif", [["gasherbrum"]]),
    ("skardu-valley", [["skardu"]]),
    ("hunza-valley-tours", [["hunza"]]),
    ("islamabad-city-tour-packages-route-and-planner", [["islamabad city"], ["islamabad", "city tour"]]),
    ("zanzibar-island-experience-packages-route-and-planner", [["zanzibar"]]),
    ("tanzania-packages-route-and-planner", [["tanzania"], ["arusha"], ["kilimanjaro"]]),
    ("bhaktapur-tour-packages-route-and-planner", [["bhaktapur"]]),
    ("patan-tour-packages-route-and-planner", [["patan"]]),
    ("kathmandu-valley-tour-heritage-temples-and-living-culture", [["kathmandu valley"], ["kathmandu", "heritage"]]),
]


TRIP_STOPWORDS = {
    "package", "packages", "route", "planner", "and", "with", "the", "a", "an",
    "of", "in", "at", "to", "from", "for", "by", "via", "trek", "trekking",
    "tour", "tours", "travel", "experience", "packages route planner"
}


def trip_match_name(trip_slug):
    """
    Converts a schema trip slug into a cleaner matching phrase.
    Example:
    everest-base-camp-trek-with-helicopter-return-packages-route-and-planner
    -> everest base camp trek with helicopter return
    """
    value = normalize_key(trip_slug)
    value = re.sub(r"\bpackages?\b", " ", value)
    value = re.sub(r"\broute\b", " ", value)
    value = re.sub(r"\bplanner\b", " ", value)
    value = re.sub(r"\s+", " ", value).strip()
    return value


def canonical_trip(value):
    """
    Returns a valid trip slug if the supplied value can be safely matched
    to one item in VALID_TRIPS. Otherwise returns None.
    """
    if not value:
        return None

    raw = str(value).strip()
    if raw in VALID_TRIPS:
        return raw

    raw_norm = normalize_key(raw)
    for trip in VALID_TRIPS:
        if raw_norm == normalize_key(trip) or raw_norm == trip_match_name(trip):
            return trip

    # Allow old display-like values or hand-written names to map through aliases.
    for key, trip in TRIP_ALIASES.items():
        if normalize_key(key) == raw_norm and trip in VALID_TRIPS:
            return trip

    return None


def phrase_group_present(source_norm, phrase_group):
    return all(normalize_key(phrase) in source_norm for phrase in phrase_group)


def extract_declared_trip(text):
    """
    If the source document explicitly contains a schema trip value, trust it.
    Supports:
    Trip: annapurna-base-camp-trek-packages-route-and-planner
    Trips: annapurna-base-camp-trek-packages-route-and-planner
    """
    if not text:
        return None

    for m in re.finditer(r"(?im)^\s*(?:trip|trips|valid\s*trip|matched\s*trip)\s*:\s*(.+?)\s*$", text):
        raw_value = m.group(1).strip()
        # Split only common separators used for lists.
        for part in re.split(r"\s*,\s*|\s*\|\s*", raw_value):
            trip = canonical_trip(part)
            if trip:
                return trip

    return None


def trip_token_score(trip_slug, title_norm, text_norm):
    """
    Scores a valid trip slug against the package title and full package text.
    Title matches receive much higher weight than body-text matches so a
    random itinerary mention does not overpower the package theme.
    """
    match_name = trip_match_name(trip_slug)
    tokens = [
        tok for tok in match_name.split()
        if len(tok) > 2 and tok not in TRIP_STOPWORDS
    ]

    if not tokens:
        return 0

    score = 0

    # Strong exact phrase in title.
    if match_name and match_name in title_norm:
        score += 120 + len(match_name)

    # Important: all specific tokens in title is a strong match.
    title_token_hits = 0
    text_token_hits = 0

    for token in tokens:
        token_re = rf"(^|\s){re.escape(token)}($|\s)"
        if re.search(token_re, title_norm):
            title_token_hits += 1
            score += 12
        elif re.search(token_re, text_norm):
            text_token_hits += 1
            score += 3

    if title_token_hits == len(tokens):
        score += 80

    # Specific route words should beat generic region words.
    specific_terms = {
        "heli", "helicopter", "jiri", "gokyo", "kongma", "cho", "renjo",
        "ama", "dablam", "island", "sleep", "budget", "luxury", "short",
        "road", "mardi", "khopra", "muldai", "tilicho", "poon", "ghorepani",
        "upper", "lower", "mustang", "dolpo", "rara", "khaptad", "ganja",
        "tashi", "lapcha", "gondogoro", "spantik", "laila", "pastore"
    }
    score += sum(10 for token in tokens if token in specific_terms and re.search(rf"(^|\s){re.escape(token)}($|\s)", title_norm))

    # Penalize broad candidates when only body text matches them.
    if title_token_hits == 0 and text_token_hits > 0:
        score -= 15

    return score


def canonical_subcategory(value):
    if not value:
        return None
    value = str(value).strip()
    if value in VALID_SUBCATEGORIES:
        return value
    return SUBCATEGORY_ALIASES.get(value)


def canonical_destination(value):
    if not value:
        return None
    value = str(value).strip()
    if value in VALID_DESTINATIONS:
        return value
    return DESTINATION_ALIASES.get(value)


# ------------------------------------------------------------
# 2C. STRICT SCHEMA ENUM GUARD
# ------------------------------------------------------------
# IMPORTANT RULE:
# The program may choose from the lists below, but it must never create
# a new value for trips, subcategories, or destinations.
# Final JSON output is hard-filtered so these fields contain only exact
# values already present in VALID_TRIPS, VALID_SUBCATEGORIES, and
# VALID_DESTINATIONS.

STRICT_SCHEMA_FIELDS = {
    "trips": VALID_TRIPS,
    "subcategories": VALID_SUBCATEGORIES,
    "destinations": VALID_DESTINATIONS,
}


def keep_only_valid_schema_values(values, valid_values):
    """
    Keeps only exact values found in the supplied valid_values list.

    This function does NOT create slugs, does NOT guess new enum values,
    and does NOT allow display labels through. Anything not already inside
    the predefined schema list is dropped from the output.
    """
    if values is None:
        return []

    if isinstance(values, str):
        values = [values]

    valid_set = set(valid_values)
    output = []
    seen = set()

    for value in values:
        if value in valid_set and value not in seen:
            output.append(value)
            seen.add(value)

    return output


def normalize_itinerary_text(value):
    return re.sub(r"\s+", " ", str(value or "").strip()).lower()


def dedupe_activity_list(activities):
    if not isinstance(activities, list):
        return None
    seen = set()
    output = []
    for activity in activities:
        clean_activity = str(activity or "").strip()
        key = normalize_itinerary_text(clean_activity)
        if not key or key in seen:
            continue
        seen.add(key)
        output.append(clean_activity)
    return output or None


def itinerary_completeness_score(item):
    score = 0
    if normalize_itinerary_text(item.get("description")):
        score += 100
    try:
        if item.get("elevation") is not None:
            float(item.get("elevation"))
            score += 20
    except (TypeError, ValueError):
        pass
    if isinstance(item.get("geopoint"), dict):
        score += 10
    if normalize_itinerary_text(item.get("meals")):
        score += 5
    if normalize_itinerary_text(item.get("accommodation")):
        score += 5
    if isinstance(item.get("activities"), list):
        score += len(item.get("activities"))
    if normalize_itinerary_text(item.get("title")):
        score += 1
    return score


def merge_itinerary_items(existing, incoming):
    keep_incoming = itinerary_completeness_score(incoming) > itinerary_completeness_score(existing)
    base = dict(incoming if keep_incoming else existing)
    other = existing if keep_incoming else incoming
    base["activities"] = dedupe_activity_list(base.get("activities")) or dedupe_activity_list(other.get("activities"))
    return base


def normalize_itineraries(itineraries):
    if not isinstance(itineraries, list):
        return None
    seen = set()
    by_day_number = {}
    no_day_number = []
    for item in itineraries:
        if not isinstance(item, dict):
            continue
        item = dict(item)
        item["activities"] = dedupe_activity_list(item.get("activities"))
        try:
            day_number = int(item.get("dayNumber"))
        except (TypeError, ValueError):
            day_number = None
        if day_number and day_number > 0:
            item["dayNumber"] = day_number
            by_day_number[day_number] = merge_itinerary_items(by_day_number[day_number], item) if day_number in by_day_number else item
            continue
        key = "|".join([str(item.get("dayNumber")), normalize_itinerary_text(item.get("title")), normalize_itinerary_text(item.get("caption")), normalize_itinerary_text(item.get("description")), normalize_itinerary_text(item.get("start")), normalize_itinerary_text(item.get("end"))])
        if key in seen:
            continue
        seen.add(key)
        no_day_number.append(item)
    output = [by_day_number[k] for k in sorted(by_day_number)] + no_day_number
    if not output:
        return None
    for item in output:
        item.pop("roles", None)
    if len(output) == 1:
        output[0]["roles"] = ["final_destination"]
    else:
        output[0]["roles"] = ["starting_point"]
        output[-1]["roles"] = ["final_destination"]
    return output


def enforce_strict_schema_choices(package):
    """
    Final safety gate before JSON cleanup.

    Guarantees:
    - package['trips'] contains only exact values from VALID_TRIPS
    - package['subcategories'] contains only exact values from VALID_SUBCATEGORIES
    - package['destinations'] contains only exact values from VALID_DESTINATIONS

    If a matcher accidentally returns a generated/invalid value, it is removed.
    """
    if not isinstance(package, dict):
        return package

    package["trips"] = keep_only_valid_schema_values(package.get("trips"), VALID_TRIPS) or None
    package["subcategories"] = keep_only_valid_schema_values(package.get("subcategories"), VALID_SUBCATEGORIES) or None
    package["destinations"] = keep_only_valid_schema_values(package.get("destinations"), VALID_DESTINATIONS) or None
    package["itineraries"] = normalize_itineraries(package.get("itineraries"))

    return package

# ------------------------------------------------------------
# 3. FILE READER
# ------------------------------------------------------------

def upload_document():
    if files is None:
        raise RuntimeError("google.colab.files is not available. In local Python, call read_document(path) directly.")
    uploaded = files.upload()
    if not uploaded:
        raise ValueError("No file uploaded.")
    return list(uploaded.keys())[0]


def read_txt(path):
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        return f.read()


def read_docx(path):
    import docx
    doc = docx.Document(path)
    return "\n".join(p.text for p in doc.paragraphs if p.text and p.text.strip())


def read_pdf(path):
    import pdfplumber
    text_parts = []
    with pdfplumber.open(path) as pdf:
        for page in pdf.pages:
            text_parts.append(page.extract_text() or "")
    return "\n".join(text_parts)


def read_document(path):
    ext = os.path.splitext(path)[1].lower()
    if ext == ".txt":
        return read_txt(path)
    if ext == ".docx":
        return read_docx(path)
    if ext == ".pdf":
        return read_pdf(path)
    raise ValueError("Unsupported file type. Upload .txt, .docx, or .pdf")

# ------------------------------------------------------------
# 4. NORMALIZATION HELPERS
# ------------------------------------------------------------

SECTION_HEADINGS = [
    "Trip Overview",
    "The Invitation",
    "Experiences",
    "The Journey",
    "What's Included",
    "What You Need to Arrange",
    "What You'll Need to Arrange",
    "Equipment",
    "Safety & Guide Judgment",
    "Physical Preparation",
    "Common Curiosities",
]

DAY_LABELS = {"start", "via", "end", "duration", "transport", "accommodation", "meals", "note"}


def clean_text(text):
    if not text:
        return ""
    replacements = {
        "\r\n": "\n",
        "\r": "\n",
        "’": "'",
        "‘": "'",
        "“": '"',
        "”": '"',
        "–": "-",
        "—": "—",
        "•": "●",
    }
    for old, new in replacements.items():
        text = text.replace(old, new)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


# ------------------------------------------------------------
# SHORT NAME + SUBCATEGORY INFERENCE LOGIC
# Replace your old normalize_key, make_short_name, and infer_subcategories
# with this complete block.
# ------------------------------------------------------------

import re


def normalize_key(value):
    """
    Normalizes text for keyword matching.
    Keeps words readable and removes punctuation problems from PDF extraction.
    """
    if value is None:
        return ""

    value = str(value)
    value = value.replace("’", "'").replace("“", '"').replace("”", '"')
    value = value.replace("—", " ").replace("–", " ").replace("-", " ")
    value = value.lower()
    value = re.sub(r"[^a-z0-9\s/&+().]", " ", value)
    value = re.sub(r"\s+", " ", value).strip()
    return value


def make_short_name(name):
    """
    Creates better short names while preserving the package theme.

    Examples:
    Responsible Everest Base Camp Trek -> Responsible EBC Trek
    Luxury Everest Base Camp Trek -> Luxury EBC Trek
    Budget Everest Base Camp Trek -> Budget EBC Trek
    Everest Base Camp Trek with Helicopter Return -> EBC Trek with Heli Return
    Annapurna Base Camp Trek -> ABC Trek
    Annapurna Circuit Mountain Bike Tour -> Annapurna Circuit MTB Tour
    Kanchenjunga North Base Camp Trek -> Kanchenjunga North BC Trek
    """
    if not name:
        return None

    short = str(name).strip()

    # Remove SEO/package suffixes
    suffix_patterns = [
        r"\s*—\s*Packages,\s*Route\s*&\s*Planner\s*$",
        r"\s*-\s*Packages,\s*Route\s*&\s*Planner\s*$",
        r"\s*—\s*Heritage,\s*Temples\s*&\s*Living\s*Culture\s*$",
    ]

    for pattern in suffix_patterns:
        short = re.sub(pattern, "", short, flags=re.I).strip()

    # Specific replacements first
    replacements = [
        # Everest / EBC
        (r"\bEverest Base Camp\s*\(EBC\)\b", "EBC"),
        (r"\bEverest Base Camp\b", "EBC"),

        # Annapurna / ABC
        (r"\bAnnapurna Base Camp\b", "ABC"),
        (r"\bAnnapurna Circuit Trekking\b", "Annapurna Circuit Trek"),
        (r"\bAnnapurna Circuit Mountain Bike Tour\b", "Annapurna Circuit MTB Tour"),
        (r"\bAnnapurna Circuit Mountain Biking\b", "Annapurna Circuit MTB"),

        # Other base camps
        (r"\bKanchenjunga North Base Camp\b", "Kanchenjunga North BC"),
        (r"\bKanchenjunga South Base Camp\b", "Kanchenjunga South BC"),
        (r"\bKanchenjunga Base Camp\b", "Kanchenjunga BC"),
        (r"\bMakalu Base Camp\b", "Makalu BC"),
        (r"\bK2 Base Camp\b", "K2 BC"),
        (r"\bDhaulagiri Base Camp\b", "Dhaulagiri BC"),
        (r"\bAma Dablam Base Camp\b", "Ama Dablam BC"),
        (r"\bPastore Base Camp\b", "Pastore BC"),
        (r"\bLaila Peak Base Camp\b", "Laila Peak BC"),
        (r"\bApi Himal Base Camp\b", "Api Himal BC"),

        # Common terms
        (r"\bMountain Bike\b", "MTB"),
        (r"\bMountain Biking\b", "MTB"),
        (r"\bHelicopter Return\b", "Heli Return"),
        (r"\bHelicopter\b", "Heli"),
        (r"\bNational Park\b", "NP"),
        (r"\bTrekking\b", "Trek"),
    ]

    for pattern, replacement in replacements:
        short = re.sub(pattern, replacement, short, flags=re.I)

    # Fix duplicate EBC forms like "EBC (EBC) Trek"
    short = re.sub(r"\bEBC\s*\(EBC\)", "EBC", short, flags=re.I)

    # Clean double words/spaces
    short = re.sub(r"\bTrek Trek\b", "Trek", short, flags=re.I)
    short = re.sub(r"\bTour Tour\b", "Tour", short, flags=re.I)
    short = re.sub(r"\s+", " ", short).strip()

    # Avoid cutting too aggressively, but keep admin-friendly length
    if len(short) > 60:
        short = short[:60].rstrip()

    return short


def infer_subcategories(title, text):
    """
    Infers package subcategories using weighted keyword matching.

    Important:
    - Only returns categories that exist inside VALID_SUBCATEGORIES.
    - Normal EBC treks are NOT classified as expeditions just because they contain Everest.
    - Expedition classification requires stronger words like expedition, peak climbing,
      summit attempt, 7000m, 8000m, K2 expedition, Spantik, etc.
    """
    t = normalize_key(f"{title} {text}")
    scores = {}

    def add(category, points):
        category = canonical_subcategory(category)
        if category in VALID_SUBCATEGORIES:
            scores[category] = scores.get(category, 0) + points

    def has_any(keywords):
        return any(normalize_key(k) in t for k in keywords)

    # ------------------------------------------------------------
    # 1. City Day Tours & Sightseeing
    # ------------------------------------------------------------
    if has_any([
        "city tour", "sightseeing", "heritage", "temple", "living culture",
        "durbar square", "monastery visit", "cultural tour", "old city",
        "bhaktapur", "patan", "kathmandu valley tour", "islamabad city",
        "janakpur", "lumbini", "stupa", "boudhanath", "swayambhunath",
        "pashupatinath", "palace", "museum", "bazaar walk"
    ]):
        add("City Day Tours & Sightseeing", 6)

    # ------------------------------------------------------------
    # 2. Expeditions 7000–8000m
    # ------------------------------------------------------------
    if has_any([
        "expedition", "peak climbing", "summit attempt", "summit push",
        "fixed rope", "base camp rotation", "high camp", "camp i", "camp ii",
        "camp iii", "camp iv", "7000", "8000", "8611", "8126", "8167",
        "nanga parbat", "spantik", "gasherbrum", "k2 expedition",
        "mount k2", "dhampus peak", "island peak climbing", "mera peak climbing",
        "pachermo peak", "yalung ri", "technical climbing", "climbing permit"
    ]):
        add("Expeditions (7000-8000m)", 7)
        add("Land & Vertical", 3)

    # ------------------------------------------------------------
    # 3. Bike & Moto
    # ------------------------------------------------------------
    if has_any([
        "bike", "biking", "mountain bike", "mountain biking", "mtb",
        "cycling", "cycle", "moto", "motorbike", "off road ride",
        "offroad", "upper mustang bike", "annapurna circuit mountain bike",
        "lower mustang mountain biking"
    ]):
        add("Bike & Moto", 7)

    # ------------------------------------------------------------
    # 4. Heli Treks & Heli Returns
    # ------------------------------------------------------------
    if has_any([
        "heli", "helicopter", "helicopter return", "heli return",
        "flightseeing", "aerial return", "fly out", "fly back by helicopter",
        "everest base camp heli trek"
    ]):
        add("Heli Treks & Heli Returns", 7)
        add("Aerial & Sky", 4)

    # ------------------------------------------------------------
    # 5. Aerial & Sky
    # ------------------------------------------------------------
    if has_any([
        "paragliding", "ultralight", "sky", "flightseeing", "mountain flight",
        "scenic flight", "aerial", "helicopter", "heli", "fly over"
    ]):
        add("Aerial & Sky", 5)

    # ------------------------------------------------------------
    # 6. Yoga Retreats
    # ------------------------------------------------------------
    if has_any([
        "yoga", "yoga trek", "retreat", "wellness retreat",
        "breathing practice", "mind body", "yoga session",
        "rara lake yoga"
    ]):
        add("Yoga Retreats", 7)
        add("Mindful Trekking & Nature Immersion", 3)

    # ------------------------------------------------------------
    # 7. Meditation & Vipassana
    # ------------------------------------------------------------
    if has_any([
        "meditation", "vipassana", "silent retreat", "mindfulness practice",
        "monastic stay", "inner stillness", "breath awareness",
        "spiritual practice"
    ]):
        add("Meditation & Vipassana", 7)
        add("Mindful Trekking & Nature Immersion", 3)

    # ------------------------------------------------------------
    # 8. Ayurveda & Detox
    # ------------------------------------------------------------
    if has_any([
        "ayurveda", "ayurvedic", "panchakarma", "detox",
        "herbal treatment", "spa therapy", "wellness detox",
        "healing therapy", "body cleansing"
    ]):
        add("Ayurveda & Detox (Panchakarma)", 7)

    # ------------------------------------------------------------
    # 9. Photography, Filming & Permits
    # ------------------------------------------------------------
    if has_any([
        "photography", "photo", "filming", "documentary", "camera",
        "cinematic", "film permit", "drone permit", "visual storytelling",
        "landscape photography", "night photography", "nocturnal",
        "astrophotography", "sunrise photography", "golden hour"
    ]):
        add("Photography, Filming & Permits", 7)

    # ------------------------------------------------------------
    # 10. Lakes & Hill Stations
    # ------------------------------------------------------------
    if has_any([
        "lake", "rara", "phoksundo", "gokyo", "tilicho", "panch pokhari",
        "ice lake", "kapuche", "khaptad", "hill station", "zanzibar",
        "beach", "island", "phewa", "tsho rolpa", "gosaikunda"
    ]):
        add("Lakes & Hill Stations", 6)

    # ------------------------------------------------------------
    # 11. Lakeside Romance Pokhara
    # ------------------------------------------------------------
    if has_any([
        "pokhara romance", "lakeside romance", "couple", "honeymoon",
        "romantic", "anniversary", "phewa lake", "lakeside pokhara",
        "romantic escape"
    ]):
        add("Lakeside Romance (Pokhara)", 7)
        add("Adventure for Two", 4)

    # ------------------------------------------------------------
    # 12. Adventure for Two
    # ------------------------------------------------------------
    if has_any([
        "for two", "couple", "honeymoon", "romantic", "private couple",
        "anniversary", "shared adventure", "two travelers"
    ]):
        add("Adventure for Two", 6)

    # ------------------------------------------------------------
    # 13. Weekend & Short Breaks
    # ------------------------------------------------------------
    if has_any([
        "short", "weekend", "day hike", "short trek", "easy escape",
        "quick escape", "nagarkot", "dhampus", "ghandruk",
        "namche bazaar trek", "chisapani", "muldai", "pikey peak",
        "ama yangri", "rara short", "short langtang", "short manaslu",
        "short everest base camp", "short makalu"
    ]):
        add("Weekend & Short Breaks", 6)

    # ------------------------------------------------------------
    # 14. Easy Treks & Nature Walks
    # ------------------------------------------------------------
    if has_any([
        "easy trek", "easy walk", "nature walk", "beginner friendly",
        "gentle trail", "low altitude", "family friendly", "soft trek",
        "village walk", "forest walk", "first time trekker",
        "beginner preparation", "beginner focused", "easy trekking"
    ]):
        add("Easy Treks & Nature Walks", 6)

    # ------------------------------------------------------------
    # 15. Mountain & Sunrise Escapes
    # ------------------------------------------------------------
    if has_any([
        "sunrise", "viewpoint", "view point", "panorama", "poon hill",
        "mohare danda", "khopra ridge", "muldai viewpoint",
        "muldai view point", "mardi himal", "everest panorama",
        "annapurna panorama", "mountain view", "himalayan view",
        "ridge view", "dawn view"
    ]):
        add("Mountain & Sunrise Escapes", 6)

    # ------------------------------------------------------------
    # 16. Classic & Short Treks
    # ------------------------------------------------------------
    if has_any([
        "trek", "trekking", "base camp", "trail", "pass", "valley",
        "circuit", "sanctuary", "traverse", "heritage trek",
        "gokyo", "langtang", "manaslu", "annapurna", "dolpo",
        "mustang", "kanchenjunga", "makalu", "rolwaling",
        "khumbu", "rara lake trek", "jomsom muktinath",
        "nar phu", "tamang heritage", "tiji festival"
    ]):
        add("Classic & Short Treks", 5)

    # ------------------------------------------------------------
    # 17. Mindful Trekking & Nature Immersion
    # ------------------------------------------------------------
    if has_any([
        "mindful", "pacing", "acclimatization", "nature immersion",
        "awareness", "mental readiness", "slow travel", "responsible",
        "low impact", "trail discipline", "culture interpretation",
        "etiquette", "sherpa culture", "buddhist etiquette", "reflection",
        "learning", "altitude education", "health awareness",
        "guide judgment", "safety communication", "responsible trail behavior",
        "hygiene adjustment", "body check"
    ]):
        add("Mindful Trekking & Nature Immersion", 6)

    # ------------------------------------------------------------
    # 18. Land & Vertical
    # ------------------------------------------------------------
    if has_any([
        "pass", "la pass", "high pass", "three passes", "ganja la",
        "tashi lapcha", "mesokanto la", "gondogoro la", "technical",
        "scramble", "ridge", "vertical", "climbing", "steep ascent",
        "glacier", "ice", "roped", "crevasse", "high route",
        "crossing", "mountain pass", "peak", "summit"
    ]):
        add("Land & Vertical", 6)

    # ------------------------------------------------------------
    # 19. Water & Canyon
    # ------------------------------------------------------------
    if has_any([
        "canyon", "canyoning", "waterfall", "rafting", "river",
        "kayak", "kayaking", "water activity", "whitewater",
        "white water", "stream crossing"
    ]):
        add("Water & Canyon", 6)

    # ------------------------------------------------------------
    # 20. Extensions & Multi-Country
    # ------------------------------------------------------------
    if has_any([
        "extension", "add on", "multi country", "nepal and",
        "pakistan and", "tanzania and", "zanzibar extension",
        "cross border", "combine with", "plus upper mustang",
        "dolpo mustang traverse", "dolpo rara traverse",
        "dhualagiri plus upper mustang", "nar phu to upper mustang"
    ]):
        add("Extensions & Multi-Country", 6)

    # ------------------------------------------------------------
    # Extra protection rules
    # ------------------------------------------------------------

    # If package is a city tour, city category should usually be first.
    if has_any(["city tour", "sightseeing", "heritage", "bhaktapur tour", "patan tour", "islamabad city tour"]):
        add("City Day Tours & Sightseeing", 5)

    # If package is a normal Everest Base Camp trek, do not force Expedition.
    normal_ebc_terms = [
        "everest base camp trek",
        "ebc trek",
        "budget everest base camp",
        "luxury everest base camp",
        "responsible everest base camp",
        "short everest base camp",
        "sleep at everest base camp"
    ]

    if has_any(normal_ebc_terms) and not has_any([
        "expedition", "peak climbing", "summit attempt", "fixed rope",
        "base camp rotation", "high camp", "camp i", "camp ii", "camp iii"
    ]):
        if "Expeditions (7000-8000m)" in scores:
            scores["Expeditions (7000-8000m)"] = max(0, scores["Expeditions (7000-8000m)"] - 7)

    # Sort by score
    ordered = sorted(scores.items(), key=lambda x: x[1], reverse=True)

    # Keep strong matches only
    result = [category for category, score in ordered if score >= 5]

    # Limit to top 3 subcategories
    return result[:3]

def enum_base(value):
    return normalize_key(value)


def dedupe_preserve_order(items):
    output = []
    seen = set()
    for item in items:
        if item is None:
            continue
        key = normalize_key(item)
        if key and key not in seen:
            output.append(item)
            seen.add(key)
    return output


def split_lines(section_text):
    return [line.strip() for line in clean_text(section_text).splitlines() if line.strip()]


def label_value(text, label):
    # Supports line-wrapped values. Stops at another Known Label: or section heading.
    text = clean_text(text)
    label_re = re.escape(label)
    m = re.search(rf"(?im)^\s*{label_re}\s*:\s*(.+?)(?=\n\s*[A-Z][A-Za-z &/'-]{{1,45}}\s*:|\n\s*(?:" + "|".join(map(re.escape, SECTION_HEADINGS)) + r")\s*$|\Z)", text, flags=re.DOTALL)
    if not m:
        return ""
    value = " ".join(split_lines(m.group(1)))
    return value.strip()


def find_first_label_value(text, labels):
    for label in labels:
        value = label_value(text, label)
        if value:
            return value
    return ""


def extract_section(text, heading, next_headings=None):
    text = clean_text(text)
    heading_variants = {
        "What's Included": ["What's Included", "What’s Included"],
        "What You'll Need to Arrange": ["What You'll Need to Arrange", "What You’ll Need to Arrange", "What You Need to Arrange"],
    }
    variants = heading_variants.get(heading, [heading])

    start_match = None
    for variant in variants:
        m = re.search(rf"(?im)^\s*{re.escape(variant)}\s*$", text)
        if m and (start_match is None or m.start() < start_match.start()):
            start_match = m
    if not start_match:
        return ""

    if next_headings is None:
        next_headings = SECTION_HEADINGS
    start_idx = start_match.end()
    end_idx = len(text)

    possible_next = []
    for nh in next_headings:
        if normalize_key(nh) == normalize_key(heading):
            continue
        for variant in heading_variants.get(nh, [nh]):
            m = re.search(rf"(?im)^\s*{re.escape(variant)}\s*$", text[start_idx:])
            if m:
                possible_next.append(start_idx + m.start())
    if possible_next:
        end_idx = min(possible_next)
    return text[start_idx:end_idx].strip()


def merge_bullets(section_text):
    """PDF extraction often wraps bullets. This merges wrapped lines into the same bullet."""
    lines = split_lines(section_text)
    bullets = []
    current = None

    for line in lines:
        is_bullet = bool(re.match(r"^\s*(?:●|•|-|\*)\s+", line))
        stripped = re.sub(r"^\s*(?:●|•|-|\*)\s+", "", line).strip()
        if not stripped:
            continue

        if is_bullet:
            if current:
                bullets.append(current.strip())
            current = stripped
        else:
            if current:
                current += " " + stripped
            else:
                # Some documents may paste lists without bullets. Keep each line as a list item.
                bullets.append(stripped)

    if current:
        bullets.append(current.strip())

    return [re.sub(r"\s+", " ", b).strip() for b in bullets if b.strip()]


def section_as_paragraphs(section_text):
    section_text = clean_text(section_text)
    paragraphs = []
    current = []
    for line in section_text.splitlines():
        line = line.strip()
        if not line:
            if current:
                paragraphs.append(" ".join(current).strip())
                current = []
        else:
            current.append(line)
    if current:
        paragraphs.append(" ".join(current).strip())
    return "\n\n".join(paragraphs).strip()

# ------------------------------------------------------------
# 5. BASIC FIELD EXTRACTION
# ------------------------------------------------------------


def extract_duration(value):
    if not value:
        return None
    m = re.search(r"\b(\d{1,3})\b", str(value))
    return int(m.group(1)) if m else None


def infer_difficulty(text):
    t = normalize_key(text)
    if any(k in t for k in ["extremely difficult", "extreme", "expedition", "technical climbing", "7000", "8000", "8611"]):
        return "extremely_difficult"
    if any(k in t for k in ["strenuous", "difficult", "high altitude", "pass", "base camp", "peak climbing"]):
        return "difficult"
    if "moderate" in t:
        return "moderate"
    if any(k in t for k in ["easy", "beginner", "nature walk", "city tour", "short walk"]):
        return "beginner"
    return None


def infer_seasons(text):
    t = normalize_key(text)
    seasons = []
    for s in VALID_SEASONS:
        if re.search(rf"\b{re.escape(s)}\b", t):
            seasons.append(s)
    # Common trekking phrasing: spring and autumn are preferred.
    return dedupe_preserve_order(seasons)


# ------------------------------------------------------------
# ACCOMMODATION ENUM + DEPARTURE DATE HELPERS
# ------------------------------------------------------------

VALID_ACCOMMODATIONS = ["teahouselodge", "teahouse", "lodge", "hotel"]


def normalize_accommodation_enum(value):
    """
    Converts extracted accommodation text into the strict schema enum.

    Allowed outputs only:
    - teahouselodge
    - teahouse
    - lodge
    - hotel

    Examples:
    Teahouse / Lodge -> teahouselodge
    Teahouse-Lodge -> teahouselodge
    Hotel in Kathmandu -> hotel
    Guesthouse / Lodge -> lodge
    """
    if value is None:
        return None

    raw = str(value).strip()
    if not raw:
        return None

    normalized = normalize_key(raw)
    normalized = normalized.replace("tea house", "teahouse")
    compact = re.sub(r"[^a-z0-9]+", "", normalized)

    # Already valid enum values should pass through unchanged.
    if compact in VALID_ACCOMMODATIONS:
        return compact

    # Combined teahouse + lodge must use the exact enum: teahouselodge.
    if "teahouse" in compact and "lodge" in compact:
        return "teahouselodge"

    # Common combined wording after PDF / DOCX extraction.
    if re.search(r"\bteahouse\s*(?:/|and|&|-)?\s*lodge\b", normalized):
        return "teahouselodge"

    if "teahouse" in compact:
        return "teahouse"

    if "lodge" in compact or "guesthouse" in compact or "guesthouse" in normalized.replace(" ", ""):
        return "lodge"

    if "hotel" in compact or "resort" in compact:
        return "hotel"

    # If the source uses unsupported accommodation words like camp/tent/homestay,
    # do not output an invalid slug. Null is safer than failing schema validation.
    return None


DEFAULT_DEPARTURE_YEAR = 2026
DEFAULT_DEPARTURE_DATE = "2026-09-01T00:00:00.000Z"

SEASON_DEPARTURE_DATES = {
    "spring": "2026-03-01T00:00:00.000Z",
    "summer": "2026-06-01T00:00:00.000Z",
    "autumn": "2026-09-01T00:00:00.000Z",
    "winter": "2026-12-01T00:00:00.000Z",
}


def normalize_iso_departure_date(value):
    """
    Returns a valid UTC ISO date string in the required format.
    Accepts:
    - 2026-09-01T00:00:00.000Z
    - 2026-09-01
    Invalid or empty values return None.
    """
    if value is None:
        return None
    value = str(value).strip()
    if not value:
        return None

    if re.fullmatch(r"\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:\d{2}\.\d{3}Z", value):
        return value

    if re.fullmatch(r"\d{4}-\d{2}-\d{2}", value):
        return f"{value}T00:00:00.000Z"

    return None


def resolve_departure_dates(manual_dates=None, seasons=None):
    """
    Ensures departureDates is never empty.

    Priority:
    1. Use valid MANUAL_INPUTS departureDates if supplied.
    2. Generate ISO dates from inferred seasons.
       spring -> 2026-03-01T00:00:00.000Z
       summer -> 2026-06-01T00:00:00.000Z
       autumn -> 2026-09-01T00:00:00.000Z
       winter -> 2026-12-01T00:00:00.000Z
    3. If no season is detected, default to 2026-09-01T00:00:00.000Z.
    """
    output = []

    if manual_dates:
        if isinstance(manual_dates, str):
            manual_dates = [manual_dates]
        for value in manual_dates:
            iso_value = normalize_iso_departure_date(value)
            if iso_value:
                output.append(iso_value)

    if not output:
        for season in seasons or []:
            key = normalize_key(season)
            iso_value = SEASON_DEPARTURE_DATES.get(key)
            if iso_value:
                output.append(iso_value)

    if not output:
        output.append(DEFAULT_DEPARTURE_DATE)

    return dedupe_preserve_order(output)


def make_short_name(name):
    if not name:
        return None
    name = re.sub(r"\s+", " ", name).strip()
    name = re.sub(r"\s*[—-]\s*Packages, Route\s*&\s*Planner\s*$", "", name, flags=re.I)
    name = re.sub(r"\s*[—-]\s*Packages, Route\s*and\s*Planner\s*$", "", name, flags=re.I)
    # Short but still readable.
    return name[:80].strip()


def make_meta_title(name, duration):
    if name and duration:
        return f"{name} | {duration}-Day Guided Package"
    if name:
        return f"{name} | Guided Package"
    return None


def make_meta_description(name, duration, difficulty, destinations):
    bits = []
    if duration:
        bits.append(f"A {duration}-day")
    else:
        bits.append("A guided")
    bits.append((name or "travel package").strip())
    if difficulty:
        bits.append(f"with {difficulty.replace('_', ' ')} difficulty")
    if destinations:
        bits.append(f"covering {', '.join(destinations[:4])}")
    return re.sub(r"\s+", " ", " ".join(bits)).strip() + "."

# ------------------------------------------------------------
# 6. COORDINATE AND ELEVATION EXTRACTION
# ------------------------------------------------------------


def parse_coordinate_from_line(line):
    pattern = r"(-?\d+(?:\.\d+)?)\s*°?\s*([NS])\s*,\s*(-?\d+(?:\.\d+)?)\s*°?\s*([EW])"
    m = re.search(pattern, line or "", flags=re.I)
    if not m:
        return None
    lat = float(m.group(1))
    lon = float(m.group(3))
    if m.group(2).upper() == "S":
        lat = -lat
    if m.group(4).upper() == "W":
        lon = -lon
    return {"latitude": lat, "longitude": lon}


def extract_elevation(line):
    if not line:
        return None
    # Handles: 5,545-5,644 m, around 5,364 m, approx. 27.6971 N ... around 1,338 m
    elevation_part = str(line)
    if "around" in elevation_part.lower():
        elevation_part = re.split(r"around", elevation_part, flags=re.I)[-1]
    nums = re.findall(r"\b(\d{1,3}(?:,\d{3})+|\d{3,5})\b\s*(?=m\b|[-–—])?", elevation_part)
    clean_nums = []
    for n in nums:
        try:
            val = int(n.replace(",", ""))
            # Avoid taking latitude/longitude values as elevation.
            if 100 <= val <= 9000:
                clean_nums.append(val)
        except ValueError:
            pass
    return max(clean_nums) if clean_nums else None


def all_coordinate_lines(text):
    output = []
    for line in split_lines(text):
        coord = parse_coordinate_from_line(line)
        if coord:
            output.append({
                "line": line,
                "coordinate": coord,
                "elevation": extract_elevation(line),
            })
    return output


def infer_package_coordinates(text):
    points = all_coordinate_lines(text)
    if not points:
        return {"latitude": None, "longitude": None}

    # Prefer highest meaningful route elevation because this often represents the main target/high point.
    with_elevation = [p for p in points if p["elevation"] is not None]
    if with_elevation:
        chosen = max(with_elevation, key=lambda p: p["elevation"])
        return chosen["coordinate"]

    return points[0]["coordinate"]

# ------------------------------------------------------------
# 7. TRIP, DESTINATION, SUBCATEGORY INFERENCE
# ------------------------------------------------------------


def match_valid_trip(title, text):
    """
    Returns the single most specific valid trip slug from VALID_TRIPS.

    Logic:
    1. Trust an explicit Trip/Trips field if it contains a valid schema slug.
    2. Apply ordered high-specificity route rules first.
    3. Use exact/alias matching.
    4. Use weighted scoring, with title matches prioritized over itinerary/body text.
    5. Return None instead of guessing a weak generic trip.
    """
    title_norm = normalize_key(title)
    text_norm = normalize_key(text)
    combined_norm = normalize_key(f"{title or ''} {text or ''}")

    # 1) Explicit schema value in the document.
    declared = extract_declared_trip(text)
    if declared:
        return declared

    # 2) Ordered specific rules. Title first, then full text.
    for trip, phrase_groups in TRIP_PRIORITY_RULES:
        if trip not in VALID_TRIPS:
            continue
        if any(phrase_group_present(title_norm, group) for group in phrase_groups):
            return trip

    for trip, phrase_groups in TRIP_PRIORITY_RULES:
        if trip not in VALID_TRIPS:
            continue
        if any(phrase_group_present(combined_norm, group) for group in phrase_groups):
            return trip

    # 3) Exact clean-title match against valid trip slugs.
    for trip in VALID_TRIPS:
        if title_norm and title_norm in {normalize_key(trip), trip_match_name(trip)}:
            return trip

    # 4) Aliases.
    for key, trip in TRIP_ALIASES.items():
        if trip in VALID_TRIPS and normalize_key(key) in title_norm:
            return trip

    # 5) Weighted scoring over all valid trips.
    scored = []
    for trip in VALID_TRIPS:
        score = trip_token_score(trip, title_norm, combined_norm)
        if score > 0:
            scored.append((score, len(trip_match_name(trip)), trip))

    if scored:
        scored.sort(reverse=True)
        best_score, _, best_trip = scored[0]

        # Threshold prevents random body-text matches from becoming a trip.
        if best_score >= 35:
            return best_trip

    # 6) Conservative fuzzy fallback from title only.
    best_trip = None
    best_score = 0.0
    for trip in VALID_TRIPS:
        base = trip_match_name(trip)
        score = SequenceMatcher(None, title_norm, base).ratio()
        if score > best_score:
            best_score = score
            best_trip = trip

    if best_score >= 0.82:
        return best_trip

    return None


# Final strict wrapper around trip matcher. It guarantees that match_valid_trip
# can return only None or an exact item from VALID_TRIPS.
_original_match_valid_trip = match_valid_trip


def match_valid_trip(title, text):
    trip = _original_match_valid_trip(title, text)
    return trip if trip in VALID_TRIPS else None


DESTINATION_KEYWORDS = {
    "Rara Lake": ["rara lake", "rara"],
    "Zanzibar": ["zanzibar"],
    "Everest Region": ["everest", "khumbu", "lukla", "namche", "dingboche", "lobuche", "gorak shep", "pheriche", "tengboche", "kala patthar"],
    "Bhaktapur": ["bhaktapur"],
    "Annapurna Region": ["annapurna", "ghorepani", "poon hill", "machhapuchhre", "mardi", "khopra", "muldai", "jomsom", "muktinath", "tilicho", "manang", "ghandruk", "dhampus", "mohare", "sikles", "kapuche"],
    "Nagarkot": ["nagarkot"],
    "Dhaulagiri region": ["dhaulagiri"],
    "Kathmandu": ["kathmandu", "tribhuvan international airport", "tia"],
    "K7 Peak": ["k7"],
    "Chitwan National Park": ["chitwan"],
    "Islamabad": ["islamabad"],
    "Dolpo Region": ["dolpo", "shey", "phoksundo"],
    "Spantik Peak": ["spantik"],
    "Rolwaling Region": ["rolwaling", "tsho rolpa", "tashi lapcha", "bigu gompa", "pachermo", "yalung ri"],
    "Manaslu Region": ["manaslu", "tsum valley", "larkya", "gorkha"],
    "K2 Base Camp": ["k2 base camp", "concordia", "baltoro", "gondogoro la", "gasherbrum", "mount k2"],
    "Janakpur": ["janakpur"],
    "Langtang Region": ["langtang valley", "langtang region", "gosaikunda", "ganja la", "helambu", "ama yangri"],
    "Langtang": ["langtang"],
    "Gilgit-Baltistan": ["gilgit", "baltistan", "skardu", "hunza", "nanga parbat", "karakoram", "laila peak", "pastore", "gasherbrum", "gondogoro"],
    "Pokhara": ["pokhara", "phewa", "lakeside", "sarangkot"],
    "Patan": ["patan"],
    "Kanchenjunga region": ["kanchenjunga"],
    "Bandipur": ["bandipur"],
    "Makalu Barun region": ["makalu", "barun"],
    "Far West Region": ["far west", "far-west", "api himal", "khaptad", "bardia"],
    "Laila Peak": ["laila peak"],
    "Arusha": ["arusha", "kilimanjaro", "tanzania"],
    "Lumbini": ["lumbini"],
    "Bardia National Park": ["bardia"],
    "Upper Mustang": ["upper mustang", "lo manthang", "tiji", "mustang"],
}


def contains_keyword(normalized_text, keyword):
    key = normalize_key(keyword)
    if not key:
        return False
    return bool(re.search(rf"(^|\s){re.escape(key)}($|\s)", normalized_text))


def infer_destinations(title, text):
    title_norm = normalize_key(title)
    text_norm = normalize_key(text)
    matched = []

    # Prefer title-level matches first.
    for dest in VALID_DESTINATIONS:
        dest_clean = dest.strip()
        dest_norm = normalize_key(dest_clean)
        if dest_norm and dest_norm in title_norm:
            matched.append(dest_clean)

    # Controlled keyword matches from title + route text.
    for dest, keywords in DESTINATION_KEYWORDS.items():
        if dest not in [d.strip() for d in VALID_DESTINATIONS]:
            continue
        if any(contains_keyword(title_norm, k) or contains_keyword(text_norm, k) for k in keywords):
            matched.append(dest)

    # Avoid returning both Langtang and Langtang Region when region is the stronger route value.
    if "Langtang Region" in matched and "Langtang" in matched:
        matched = [m for m in matched if m != "Langtang"]

    # If a Pakistan mountain route matched, Gilgit-Baltistan is usually the broader destination too.
    pakistan_mountain_destinations = {"K2 Base Camp", "Spantik Peak", "Laila Peak", "Pastore Peak"}
    if any(d in matched for d in pakistan_mountain_destinations) and "Gilgit-Baltistan" not in matched:
        matched.append("Gilgit-Baltistan")

    return dedupe_preserve_order(matched)


def infer_subcategories(title, text):
    t = normalize_key(f"{title} {text}")
    subs = []

    if any(k in t for k in ["city tour", "sightseeing", "heritage", "temple", "living culture", "bhaktapur", "patan", "islamabad"]):
        subs.append("City Day Tours & Sightseeing")

    if any(k in t for k in ["expedition", "Everest", "nanga parbat", "spantik", "7000", "8000", "8611"]):
        subs.append("Expeditions (7000-8000m)")

    if any(k in t for k in ["bike", "biking", "mountain bike", "cycling", "moto"]):
        subs.append("Bike & Moto")

    if any(k in t for k in ["heli", "helicopter", "flightseeing"]):
        subs.append("Heli Treks & Heli Returns")

    if any(k in t for k in ["yoga", "retreat"]):
        subs.append("Yoga Retreats")

    if any(k in t for k in ["photography", "filming", "documentary", "camera"]):
        subs.append("Photography, Filming & Permits")

    if any(k in t for k in ["lake", "rara", "phoksundo", "gokyo", "tilicho", "pokhari", "zanzibar"]):
        subs.append("Lakes & Hill Stations")

    if any(k in t for k in ["short", "weekend", "day hike", "nature walk", "nagarkot", "dhampus", "ghandruk", "namche bazaar"]):
        subs.append("Weekend & Short Breaks")

    if any(k in t for k in ["trek", "trekking", "base camp", "trail", "pass", "valley"]):
        subs.append("Classic & Short Treks")

    if any(k in t for k in ["pacing", "mindful", "acclimatization", "nature immersion", "awareness", "mental readiness", "culture interpretation", "etiquette"]):
        subs.append("Mindful Trekking & Nature Immersion")

    return [canonical_subcategory(s) for s in dedupe_preserve_order(subs) if canonical_subcategory(s) in VALID_SUBCATEGORIES]


# ------------------------------------------------------------
# 7B. FINAL OVERRIDE: SHORT NAME + SUBCATEGORY LOGIC
# This block intentionally appears AFTER older helper definitions so it
# overrides them and prevents stale notebook/code definitions from being used.
# ------------------------------------------------------------


def make_short_name(name):
    """
    Creates clean short names while preserving the package theme.

    Examples:
    Responsible Everest Base Camp Trek -> Responsible EBC Trek
    Luxury Everest Base Camp Trek -> Luxury EBC Trek
    Budget Everest Base Camp Trek -> Budget EBC Trek
    Everest Base Camp Trek with Helicopter Return -> EBC Trek with Heli Return
    Annapurna Base Camp Trek -> ABC Trek
    Annapurna Circuit Mountain Bike Tour -> Annapurna Circuit MTB Tour
    """
    if not name:
        return None

    short = str(name).strip()

    # Remove SEO/package suffixes.
    short = re.sub(r"\s*—\s*Packages,\s*Route\s*&\s*Planner\s*$", "", short, flags=re.I)
    short = re.sub(r"\s*-\s*Packages,\s*Route\s*&\s*Planner\s*$", "", short, flags=re.I)
    short = re.sub(r"\s*—\s*Heritage,\s*Temples\s*&\s*Living\s*Culture\s*$", "", short, flags=re.I)
    short = re.sub(r"\s+", " ", short).strip()

    replacements = [
        (r"\bEverest Base Camp\s*\(EBC\)\b", "EBC"),
        (r"\bEverest Base Camp\b", "EBC"),
        (r"\bAnnapurna Base Camp\b", "ABC"),
        (r"\bKanchenjunga North Base Camp\b", "Kanchenjunga North BC"),
        (r"\bKanchenjunga South Base Camp\b", "Kanchenjunga South BC"),
        (r"\bKanchenjunga Base Camp\b", "Kanchenjunga BC"),
        (r"\bMakalu Base Camp\b", "Makalu BC"),
        (r"\bK2 Base Camp\b", "K2 BC"),
        (r"\bDhaulagiri Base Camp\b", "Dhaulagiri BC"),
        (r"\bAma Dablam Base Camp\b", "Ama Dablam BC"),
        (r"\bPastore Base Camp\b", "Pastore BC"),
        (r"\bLaila Peak Base Camp\b", "Laila Peak BC"),
        (r"\bApi Himal Base Camp\b", "Api Himal BC"),
        (r"\bAnnapurna Circuit Mountain Bike Tour\b", "Annapurna Circuit MTB Tour"),
        (r"\bAnnapurna Circuit Mountain Biking\b", "Annapurna Circuit MTB"),
        (r"\bMountain Biking\b", "MTB"),
        (r"\bMountain Bike\b", "MTB"),
        (r"\bHelicopter Return\b", "Heli Return"),
        (r"\bHelicopter\b", "Heli"),
        (r"\bNational Park\b", "NP"),
        (r"\bCircuit Trekking\b", "Circuit Trek"),
        (r"\bTrekking\b", "Trek"),
    ]

    for pattern, replacement in replacements:
        short = re.sub(pattern, replacement, short, flags=re.I)

    short = re.sub(r"\bEBC\s*\(EBC\)", "EBC", short, flags=re.I)
    short = re.sub(r"\bTrek Trek\b", "Trek", short, flags=re.I)
    short = re.sub(r"\bTour Tour\b", "Tour", short, flags=re.I)
    short = re.sub(r"\s+", " ", short).strip()

    return short[:60].rstrip() if len(short) > 60 else short


def infer_subcategories(title, text):
    """
    Infers top schema subcategories using title-first weighted scoring.

    User preference added:
    - Names containing Everest or Annapurna strongly map to
      Expeditions (7000-8000m) and Mountain & Sunrise Escapes.
    - Any photo / photography / filming package strongly maps to
      Photography, Filming & Permits.
    """
    title_text = normalize_key(title)
    full_text = normalize_key(f"{title or ''} {text or ''}")
    scores = {}

    def add(category, points):
        category = canonical_subcategory(category)
        if category in VALID_SUBCATEGORIES:
            scores[category] = scores.get(category, 0) + points

    def has_any(keywords, source=None):
        source = full_text if source is None else source
        return any(normalize_key(k) in source for k in keywords)

    # ------------------------------------------------------------
    # Strong title-level rules
    # ------------------------------------------------------------
    if has_any(["photo", "photography", "filming", "documentary", "camera", "nocturnal", "cinematic"], title_text):
        add("Photography, Filming & Permits", 15)

    if has_any(["everest", "ebc", "khumbu"], title_text):
        add("Expeditions (7000-8000m)", 14)
        add("Mountain & Sunrise Escapes", 12)
        add("Classic & Short Treks", 5)

    if has_any(["annapurna", "abc", "poon hill", "mardi himal", "khopra", "mohare", "muldai"], title_text):
        add("Expeditions (7000-8000m)", 11)
        add("Mountain & Sunrise Escapes", 13)
        add("Classic & Short Treks", 5)

    if has_any(["expedition", "peak climbing", "island peak", "mera peak", "dhampus peak", "pachermo peak", "spantik", "k2", "nanga parbat", "gasherbrum", "yalung ri", "mount k2"], title_text):
        add("Expeditions (7000-8000m)", 15)
        add("Land & Vertical", 9)

    if has_any(["bike", "biking", "mountain bike", "mtb", "cycling", "moto"], title_text):
        add("Bike & Moto", 15)

    if has_any(["heli", "helicopter"], title_text):
        add("Heli Treks & Heli Returns", 15)
        add("Aerial & Sky", 7)

    if has_any(["city tour", "bhaktapur", "patan", "kathmandu valley tour", "islamabad"], title_text):
        add("City Day Tours & Sightseeing", 15)

    if has_any(["lake", "rara", "phoksundo", "gokyo", "tilicho", "panch pokhari", "gosaikunda", "tsho rolpa", "zanzibar"], title_text):
        add("Lakes & Hill Stations", 14)

    if has_any(["yoga", "retreat"], title_text):
        add("Yoga Retreats", 15)
        add("Mindful Trekking & Nature Immersion", 7)

    if has_any(["short", "weekend", "day hike", "easy"], title_text):
        add("Weekend & Short Breaks", 12)
        add("Easy Treks & Nature Walks", 7)

    # ------------------------------------------------------------
    # Full-text rules
    # ------------------------------------------------------------
    photo_signal_count = sum(full_text.count(normalize_key(k)) for k in ["photography", "filming", "documentary", "camera", "cinematic", "astrophotography", "nocturnal"])
    if has_any(["film permit", "drone permit", "visual storytelling", "landscape photography", "night photography", "sunrise photography", "photo tour", "photo trek"]) or photo_signal_count >= 2:
        add("Photography, Filming & Permits", 12)

    if has_any(["expedition", "peak climbing", "summit attempt", "summit push", "fixed rope", "base camp rotation", "high camp", "camp i", "camp ii", "camp iii", "camp iv", "7000", "8000", "8611", "8126", "8167", "nanga parbat", "spantik", "gasherbrum", "k2 expedition", "mount k2", "technical climbing", "climbing permit"]):
        add("Expeditions (7000-8000m)", 11)
        add("Land & Vertical", 6)

    if has_any(["sunrise", "viewpoint", "view point", "panorama", "poon hill", "mohare danda", "khopra ridge", "muldai viewpoint", "muldai view point", "mardi himal", "everest panorama", "annapurna panorama", "mountain view", "himalayan view", "ridge view", "dawn view", "kala patthar"]):
        add("Mountain & Sunrise Escapes", 10)

    if has_any(["trek", "trekking", "base camp", "trail", "pass", "valley", "circuit", "sanctuary", "traverse", "heritage trek", "gokyo", "langtang", "manaslu", "annapurna", "dolpo", "mustang", "kanchenjunga", "makalu", "rolwaling", "khumbu"]):
        add("Classic & Short Treks", 6)

    if has_any(["mindful", "pacing", "acclimatization", "nature immersion", "awareness", "mental readiness", "slow travel", "responsible", "low impact", "trail discipline", "culture interpretation", "etiquette", "sherpa culture", "buddhist etiquette", "reflection", "learning", "altitude education", "health awareness", "guide judgment", "safety communication"]):
        add("Mindful Trekking & Nature Immersion", 8)

    if has_any(["bike", "biking", "mountain bike", "mountain biking", "mtb", "cycling", "cycle", "moto", "motorbike", "off road ride", "offroad"]):
        add("Bike & Moto", 12)

    if has_any(["heli trek", "heli return", "helicopter return", "everest base camp heli", "flightseeing", "aerial return", "fly out by helicopter", "fly back by helicopter", "helicopter ride", "helicopter tour"]):
        add("Heli Treks & Heli Returns", 12)
        add("Aerial & Sky", 7)

    if has_any(["paragliding", "ultralight", "flightseeing", "aerial tour", "aerial adventure", "sky activity", "helicopter ride", "helicopter tour"]):
        add("Aerial & Sky", 9)

    if has_any(["city tour", "sightseeing", "heritage", "temple", "living culture", "durbar square", "monastery visit", "cultural tour", "old city", "bhaktapur", "patan", "kathmandu valley tour", "islamabad city", "janakpur", "lumbini", "stupa", "boudhanath", "swayambhunath", "pashupatinath", "palace", "museum"]):
        add("City Day Tours & Sightseeing", 10)

    if has_any(["lake", "rara", "phoksundo", "gokyo", "tilicho", "panch pokhari", "ice lake", "kapuche", "khaptad", "hill station", "zanzibar", "beach", "island", "phewa", "tsho rolpa", "gosaikunda"]):
        add("Lakes & Hill Stations", 10)

    if has_any(["yoga", "yoga trek", "retreat", "wellness retreat", "breathing practice", "mind body", "yoga session"]):
        add("Yoga Retreats", 10)
        add("Mindful Trekking & Nature Immersion", 5)

    if has_any(["meditation", "vipassana", "silent retreat", "mindfulness practice", "monastic stay", "inner stillness", "breath awareness"]):
        add("Meditation & Vipassana", 10)
        add("Mindful Trekking & Nature Immersion", 5)

    if has_any(["ayurveda", "ayurvedic", "panchakarma", "detox", "herbal treatment", "spa therapy", "wellness detox"]):
        add("Ayurveda & Detox (Panchakarma)", 10)

    if has_any(["short", "weekend", "day hike", "short trek", "easy escape", "quick escape", "nagarkot", "dhampus", "ghandruk", "namche bazaar trek", "chisapani", "muldai", "pikey peak", "ama yangri"]):
        add("Weekend & Short Breaks", 9)

    if has_any(["easy trek", "easy walk", "nature walk", "beginner friendly", "gentle trail", "low altitude", "family friendly", "soft trek", "village walk", "forest walk", "first time trekker", "beginner preparation"]):
        add("Easy Treks & Nature Walks", 9)

    if has_any(["la pass", "high pass", "three passes", "ganja la", "tashi lapcha", "mesokanto la", "gondogoro la", "technical pass", "technical climbing", "scramble", "vertical route", "glacier crossing", "ice climbing", "roped", "crevasse", "high route", "mountain pass", "peak climbing", "summit attempt"]):
        add("Land & Vertical", 9)

    if has_any(["canyon", "canyoning", "waterfall", "rafting", "river", "kayak", "kayaking", "water activity", "whitewater", "white water"]):
        add("Water & Canyon", 9)

    if has_any(["extension", "add on", "multi country", "multi-country", "zanzibar extension", "cross border", "combine with", "plus upper mustang", "dolpo mustang traverse", "dolpo rara traverse", "nar phu to upper mustang"]):
        add("Extensions & Multi-Country", 9)

    ordered = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    result = []
    for category, score in ordered:
        if score >= 6 and category not in result:
            result.append(category)

    return result[:3]

# ------------------------------------------------------------
# 8. LOCATION FIELD INFERENCE
# ------------------------------------------------------------

LOCATION_RULES = [
    ("Kathmandu", {"country": "Nepal", "province": "Bagmati Province", "district": "Kathmandu", "cityOrVillage": "Kathmandu"}),
    ("Bhaktapur", {"country": "Nepal", "province": "Bagmati Province", "district": "Bhaktapur", "cityOrVillage": "Bhaktapur"}),
    ("Patan", {"country": "Nepal", "province": "Bagmati Province", "district": "Lalitpur", "cityOrVillage": "Patan"}),
    ("Nagarkot", {"country": "Nepal", "province": "Bagmati Province", "district": "Bhaktapur", "cityOrVillage": "Nagarkot"}),
    ("Everest Region", {"country": "Nepal", "province": "Koshi Province", "district": "Solukhumbu", "cityOrVillage": "Khumbu / Lukla"}),
    ("Annapurna Region", {"country": "Nepal", "province": "Gandaki Province", "district": "Kaski", "cityOrVillage": "Pokhara / Annapurna Region"}),
    ("Pokhara", {"country": "Nepal", "province": "Gandaki Province", "district": "Kaski", "cityOrVillage": "Pokhara"}),
    ("Dhaulagiri region", {"country": "Nepal", "province": "Gandaki Province", "district": "Myagdi", "cityOrVillage": "Dhaulagiri Region"}),
    ("Manaslu Region", {"country": "Nepal", "province": "Gandaki Province", "district": "Gorkha", "cityOrVillage": "Manaslu Region"}),
    ("Langtang Region", {"country": "Nepal", "province": "Bagmati Province", "district": "Rasuwa", "cityOrVillage": "Langtang Region"}),
    ("Langtang", {"country": "Nepal", "province": "Bagmati Province", "district": "Rasuwa", "cityOrVillage": "Langtang"}),
    ("Upper Mustang", {"country": "Nepal", "province": "Gandaki Province", "district": "Mustang", "cityOrVillage": "Upper Mustang"}),
    ("Dolpo Region", {"country": "Nepal", "province": "Karnali Province", "district": "Dolpa", "cityOrVillage": "Dolpo Region"}),
    ("Rara Lake", {"country": "Nepal", "province": "Karnali Province", "district": "Mugu", "cityOrVillage": "Rara Lake"}),
    ("Kanchenjunga region", {"country": "Nepal", "province": "Koshi Province", "district": "Taplejung", "cityOrVillage": "Kanchenjunga Region"}),
    ("Makalu Barun region", {"country": "Nepal", "province": "Koshi Province", "district": "Sankhuwasabha", "cityOrVillage": "Makalu Barun Region"}),
    ("Rolwaling Region", {"country": "Nepal", "province": "Bagmati Province", "district": "Dolakha", "cityOrVillage": "Rolwaling Region"}),
    ("Far West Region", {"country": "Nepal", "province": "Sudurpashchim Province", "district": None, "cityOrVillage": "Far West Region"}),
    ("Chitwan National Park", {"country": "Nepal", "province": "Bagmati Province", "district": "Chitwan", "cityOrVillage": "Chitwan National Park"}),
    ("Bardia National Park", {"country": "Nepal", "province": "Lumbini Province", "district": "Bardiya", "cityOrVillage": "Bardia National Park"}),
    ("Lumbini", {"country": "Nepal", "province": "Lumbini Province", "district": "Rupandehi", "cityOrVillage": "Lumbini"}),
    ("Janakpur", {"country": "Nepal", "province": "Madhesh Province", "district": "Dhanusha", "cityOrVillage": "Janakpur"}),
    ("Bandipur", {"country": "Nepal", "province": "Gandaki Province", "district": "Tanahun", "cityOrVillage": "Bandipur"}),
    ("K2 Base Camp", {"country": "Pakistan", "province": "Gilgit-Baltistan", "district": "Skardu", "cityOrVillage": "K2 Base Camp / Baltoro"}),
    ("Gilgit-Baltistan", {"country": "Pakistan", "province": "Gilgit-Baltistan", "district": None, "cityOrVillage": "Gilgit-Baltistan"}),
    ("Islamabad", {"country": "Pakistan", "province": "Islamabad Capital Territory", "district": "Islamabad", "cityOrVillage": "Islamabad"}),
    ("Spantik Peak", {"country": "Pakistan", "province": "Gilgit-Baltistan", "district": "Skardu", "cityOrVillage": "Spantik Peak"}),
    ("Laila Peak", {"country": "Pakistan", "province": "Gilgit-Baltistan", "district": "Ghanche", "cityOrVillage": "Laila Peak"}),
    ("Pastore Peak", {"country": "Pakistan", "province": "Gilgit-Baltistan", "district": "Skardu", "cityOrVillage": "Pastore Peak"}),
    ("K7 Peak", {"country": "Pakistan", "province": "Gilgit-Baltistan", "district": "Ghanche", "cityOrVillage": "K7 Peak"}),
    ("Arusha", {"country": "Tanzania", "province": "Arusha Region", "district": "Arusha", "cityOrVillage": "Arusha"}),
    ("Zanzibar", {"country": "Tanzania", "province": "Zanzibar", "district": None, "cityOrVillage": "Zanzibar"}),
]


def infer_location_fields(destinations, text):
    location = {"country": None, "province": None, "district": None, "cityOrVillage": None}

    # Prefer the first non-transit destination that appears in the inferred destination list.
    priority = [d for d in destinations if d != "Kathmandu"] + (["Kathmandu"] if "Kathmandu" in destinations else [])

    for dest in priority:
        for rule_dest, rule in LOCATION_RULES:
            if normalize_key(dest) == normalize_key(rule_dest):
                location.update(rule)
                return location

    t = normalize_key(text)
    if any(k in t for k in ["nepal", "kathmandu", "everest", "annapurna", "langtang", "manaslu"]):
        location["country"] = "Nepal"
    elif any(k in t for k in ["pakistan", "islamabad", "gilgit", "skardu", "k2"]):
        location["country"] = "Pakistan"
    elif any(k in t for k in ["tanzania", "zanzibar", "arusha"]):
        location["country"] = "Tanzania"

    return location



# ------------------------------------------------------------
# 8B. FINAL DESTINATION OVERRIDES
# Destination extraction returns schema slugs only.
# ------------------------------------------------------------

def infer_destinations(title, text):
    title_norm = normalize_key(title)
    text_norm = normalize_key(text)
    matched = []

    # Direct title-level slug matches.
    for dest in VALID_DESTINATIONS:
        dest_clean = dest.strip()
        dest_norm = normalize_key(dest_clean)
        if dest_norm and dest_norm in title_norm:
            matched.append(dest_clean)

    # Controlled keyword matches from title + route text.
    for display_dest, keywords in DESTINATION_KEYWORDS.items():
        schema_dest = canonical_destination(display_dest)
        if schema_dest not in VALID_DESTINATIONS:
            continue
        if any(contains_keyword(title_norm, k) or contains_keyword(text_norm, k) for k in keywords):
            matched.append(schema_dest)

    # Avoid returning both langtang and langtang-region when the region is stronger.
    if "langtang-region" in matched and "langtang" in matched:
        matched = [m for m in matched if m != "langtang"]

    return dedupe_preserve_order(matched)


def infer_location_fields(destinations, text):
    location = {"country": None, "province": None, "district": None, "cityOrVillage": None}

    priority = [d for d in destinations if d != "kathmandu"] + (["kathmandu"] if "kathmandu" in destinations else [])

    for dest in priority:
        for rule_dest, rule in LOCATION_RULES:
            rule_slug = canonical_destination(rule_dest)
            if normalize_key(dest) == normalize_key(rule_dest) or (rule_slug and dest == rule_slug):
                location.update(rule)
                return location

    t = normalize_key(text)
    if any(k in t for k in ["nepal", "kathmandu", "everest", "annapurna", "langtang", "manaslu"]):
        location["country"] = "Nepal"
    elif any(k in t for k in ["pakistan", "islamabad", "gilgit", "skardu", "k2"]):
        location["country"] = "Pakistan"
    elif any(k in t for k in ["tanzania", "zanzibar", "arusha"]):
        location["country"] = "Tanzania"

    return location

# ------------------------------------------------------------
# 9. ITINERARY PARSER
# ------------------------------------------------------------


def parse_place_name(line):
    if not line:
        return None
    value = re.sub(r"^\s*(Start|End|Via)\s*:\s*", "", line, flags=re.I).strip()
    value = value.split("—")[0].split("-")[0].strip()
    value = re.sub(r",\s*depending.*$", "", value, flags=re.I).strip()
    return value or None


def get_day_metadata(lines):
    data = {"start": None, "end": None, "via": [], "duration": None, "transport": None, "accommodation": None, "meals": None, "note": None}
    metadata_indices = set()
    last_label = None

    for i, line in enumerate(lines):
        m = re.match(r"^(Start|End|Via|Duration|Transport|Accommodation|Meals|Note)\s*:\s*(.*)$", line, flags=re.I)
        if m:
            label = m.group(1).lower()
            value = m.group(2).strip()
            metadata_indices.add(i)
            last_label = label
            if label == "via":
                data["via"].append(value)
            else:
                data[label] = value
            continue

        # Append only obvious continuation lines for Start/End/Via, not full prose after Accommodation.
        if last_label in {"start", "end", "via"} and re.search(r"\b(source|reference|operation|point|used|flight)\b", line, flags=re.I):
            metadata_indices.add(i)
            if last_label == "via" and data["via"]:
                data["via"][-1] = (data["via"][-1] + " " + line).strip()
            elif data[last_label]:
                data[last_label] = (data[last_label] + " " + line).strip()
        else:
            last_label = None

    return data, metadata_indices


def parse_itineraries(journey_text):
    text = clean_text(journey_text)
    if not text:
        return []

    parts = re.split(r"(?=^\s*Day\s+\d+\s*:)", text, flags=re.I | re.M)
    day_blocks = [p.strip() for p in parts if re.match(r"^\s*Day\s+\d+\s*:", p.strip(), flags=re.I)]
    itineraries = []

    for block in day_blocks:
        header = re.match(r"^\s*Day\s+(\d+)\s*:\s*(.+?)\s*$", block, flags=re.I | re.M)
        if not header:
            continue

        day_number = int(header.group(1))
        title = header.group(2).strip()
        lines = split_lines(block)
        data, metadata_indices = get_day_metadata(lines)

        start_line = data.get("start") or ""
        end_line = data.get("end") or ""
        via_lines = data.get("via") or []
        duration_line = data.get("duration") or ""
        transport_line = data.get("transport") or ""
        accommodation_line = data.get("accommodation") or ""
        meals_line = data.get("meals") or ""

        # Description is all prose after removing header and metadata lines.
        desc_lines = []
        for i, line in enumerate(lines):
            if i == 0 or i in metadata_indices:
                continue
            if re.match(r"^(Start|End|Via|Duration|Transport|Accommodation|Meals|Note)\s*:", line, flags=re.I):
                continue
            desc_lines.append(line)
        description = section_as_paragraphs("\n".join(desc_lines)) or None

        # End point is the best geopoint for the day. If no end coordinate, use final via, then start.
        geopoint = parse_coordinate_from_line(end_line)
        if not geopoint and via_lines:
            for via in reversed(via_lines):
                geopoint = parse_coordinate_from_line(via)
                if geopoint:
                    break
        if not geopoint:
            geopoint = parse_coordinate_from_line(start_line)

        # Use the highest elevation reached during the day, not only the sleeping/end point.
        elevations = []
        for candidate_line in [start_line, end_line] + via_lines:
            e = extract_elevation(candidate_line)
            if e is not None:
                elevations.append(e)
        elevation = max(elevations) if elevations else None

        activities = []
        if duration_line:
            activities.append(duration_line)
        if transport_line:
            activities.append(transport_line)
        if via_lines:
            via_names = [parse_place_name(v) for v in via_lines]
            via_names = [v for v in via_names if v]
            if via_names:
                activities.append("Via: " + ", ".join(via_names))
        if not activities:
            activities.append(title)

        # The itinerary note field is intentionally removed from JSON output.

        day_obj = {
            "dayNumber": day_number,
            "title": title,
            "caption": f"Day {day_number}: {title}",
            "mapImageUrl": None,
            "elevation": elevation,
            "description": description,
            "activities": activities,
            "meals": meals_line or None,
            "accommodation": normalize_accommodation_enum(accommodation_line),
            "geopoint": geopoint,
        }

        # roles is only allowed on the first itinerary and the last itinerary.
        # Middle itinerary objects intentionally do not include a roles field.
        if day_number == 1:
            day_obj["roles"] = ["starting_point"]

        itineraries.append(day_obj)

    if itineraries:
        # Safety cleanup: no itinerary should ever output a note field.
        for item in itineraries:
            item.pop("note", None)
            item.pop("roles", None)

        # roles is included only on the first itinerary and the last itinerary.
        itineraries[0]["roles"] = ["starting_point"]
        itineraries[-1]["roles"] = ["final_destination"]

    return itineraries

# ------------------------------------------------------------
# 10. PACKAGE SPLITTING AND EXTRACTION
# ------------------------------------------------------------


def split_package_documents(raw_text):
    text = clean_text(raw_text)
    starts = [m.start() for m in re.finditer(r"(?im)^\s*Trip Overview\s*$", text)]
    if len(starts) <= 1:
        return [text]
    parts = []
    for i, start in enumerate(starts):
        end = starts[i + 1] if i + 1 < len(starts) else len(text)
        part = text[start:end].strip()
        if part:
            parts.append(part)
    return parts


def extract_package_fields(text):
    text = clean_text(text)

    trip_overview = extract_section(text, "Trip Overview")
    invitation = extract_section(text, "The Invitation")
    experiences = extract_section(text, "Experiences")
    journey = extract_section(text, "The Journey")
    included = extract_section(text, "What's Included")
    exclusions = extract_section(text, "What You'll Need to Arrange")
    equipment = extract_section(text, "Equipment")
    safety = extract_section(text, "Safety & Guide Judgment")
    physical_prep = extract_section(text, "Physical Preparation")

    name = find_first_label_value(trip_overview or text, ["Title", "Package Title", "Name"])
    if not name:
        # Fallback: first readable non-heading line.
        for line in split_lines(text):
            if normalize_key(line) not in [normalize_key(h) for h in SECTION_HEADINGS] and ":" not in line:
                name = line.strip()
                break

    duration_text = find_first_label_value(trip_overview or text, ["Duration", "Trip Duration"])
    duration = extract_duration(duration_text)

    difficulty_text = find_first_label_value(trip_overview or text, ["Difficulty", "Difficulty Level"])
    difficulty = infer_difficulty(difficulty_text or text)

    season_text = find_first_label_value(trip_overview or text, ["Best Seasons", "Best Season", "Season"])
    seasons = infer_seasons(season_text or text)

    trip_value = match_valid_trip(name, text)

    # STRICT ENUM RULE:
    # trips, destinations, and subcategories are allowed to keep only exact
    # values from their predefined schema lists. Nothing new is generated.
    trips = keep_only_valid_schema_values([trip_value] if trip_value else [], VALID_TRIPS)
    destinations = keep_only_valid_schema_values(infer_destinations(name, text), VALID_DESTINATIONS)
    subcategories = keep_only_valid_schema_values(infer_subcategories(name, text), VALID_SUBCATEGORIES)
    loc = infer_location_fields(destinations, text)
    coords = infer_package_coordinates(text)

    highlights = merge_bullets(experiences)
    inclusions = merge_bullets(included)
    exclusion_list = merge_bullets(exclusions)
    equipment_list = merge_bullets(equipment)
    safety_list = merge_bullets(safety)
    physical_prep_list = merge_bullets(physical_prep)
    itineraries = parse_itineraries(journey or text)

    package = {
        "name": name or None,
        "shortName": make_short_name(name),
        "subcategories": subcategories or None,
        "trips": trips or None,
        "destinations": destinations or None,
        "isActive": MANUAL_INPUTS["isActive"] if MANUAL_INPUTS.get("isActive") is not None else True,
        "country": loc.get("country"),
        "province": loc.get("province"),
        "district": loc.get("district"),
        "cityOrVillage": loc.get("cityOrVillage"),
        "latitude": coords.get("latitude"),
        "longitude": coords.get("longitude"),
        "badge": MANUAL_INPUTS["badge"] if MANUAL_INPUTS.get("badge") in VALID_BADGES else "listed",
        "tier": MANUAL_INPUTS["tier"] if MANUAL_INPUTS.get("tier") in VALID_TIERS else "standard",
        "duration": duration,
        "description": section_as_paragraphs(invitation) or None,
        "paymentPolicy": MANUAL_INPUTS.get("paymentPolicy") or None,
        "difficultyLevel": difficulty,
        "pricePerPerson": str(int(float(str(MANUAL_INPUTS.get("pricePerPerson", "")).strip()))) if str(MANUAL_INPUTS.get("pricePerPerson", "")).strip() and float(str(MANUAL_INPUTS.get("pricePerPerson", "")).strip()).is_integer() else (str(float(str(MANUAL_INPUTS.get("pricePerPerson", "")).strip())) if str(MANUAL_INPUTS.get("pricePerPerson", "")).strip() else "0"),
        "currency": MANUAL_INPUTS.get("currency") or "USD",
        "tags": None,
        "highlights": highlights or None,
        "inclusions": inclusions or None,
        "exclusions": exclusion_list or None,
        "equipment": equipment_list or None,
        "safety": safety_list or None,
        "physicalPrepAbout": " ".join(physical_prep_list) if physical_prep_list else None,
        "budget": MANUAL_INPUTS["budget"] if MANUAL_INPUTS.get("budget") in VALID_BUDGETS else "midrange",
        "seasons": seasons or None,
        "departureDates": resolve_departure_dates(MANUAL_INPUTS.get("departureDates"), seasons),
        "itineraries": itineraries or None,
    }

    tags = []
    for item in [package.get("shortName"), package.get("difficultyLevel")]:
        if item:
            tags.append(str(item).replace("_", " ").title())
    for seq in [package.get("destinations"), package.get("trips"), package.get("subcategories"), package.get("seasons")]:
        if isinstance(seq, list):
            tags.extend([str(x).replace("_", " ").title() for x in seq if x])
    package["tags"] = dedupe_preserve_order(tags)[:25] if tags else None

    package = enforce_strict_schema_choices(package)
    return clean_package(package)


def clean_package(obj):
    """Remove empty optional fields, but keep schema-important fields visible as null."""
    keep_null_fields = {
        "departureDates", "country", "province", "district", "cityOrVillage",
        "latitude", "longitude", "subcategories", "trips", "destinations", "seasons",
        "mapImageUrl", "meals", "geopoint", "accommodation"
    }

    if isinstance(obj, list):
        return [clean_package(x) for x in obj]

    if isinstance(obj, dict):
        cleaned = {}
        for key, value in obj.items():
            value = clean_package(value)
            empty = value is None or value == "" or value == [] or value == {}
            if empty and key not in keep_null_fields:
                continue
            cleaned[key] = None if empty else value
        return cleaned

    return obj


def extract_packages_from_document(path_or_text, is_path=True):
    raw_text = read_document(path_or_text) if is_path else path_or_text
    packages = [extract_package_fields(part) for part in split_package_documents(raw_text)]
    return packages


In [24]:
# ------------------------------------------------------------
# RUN EXTRACTOR, SAVE JSON, AND DOWNLOAD
# ------------------------------------------------------------

print("Upload your package document: .txt, .docx, or .pdf")

if files is not None:
    input_filename = upload_document()
else:
    # For local Jupyter, replace this with your file path.
    input_filename = "package_document.txt"
    if not os.path.exists(input_filename):
        raise FileNotFoundError(
            "google.colab.files is not available. Set input_filename to your local .txt, .docx, or .pdf path."
        )

bulk_json = extract_packages_from_document(input_filename, is_path=True)

json_file = "draft_package_output.json"
with open(json_file, "w", encoding="utf-8") as f:
    json.dump(bulk_json, f, ensure_ascii=False, indent=2)

print("\n================ DRAFT JSON OUTPUT ================\n")
print(json.dumps(bulk_json, ensure_ascii=False, indent=2))

print("\n================ JSON FILE CREATED ================")
print(json_file)

# Colab download. In local Jupyter, the JSON file is saved in the current notebook folder.
if files is not None:
    files.download(json_file)
else:
    print(f"Saved locally: {os.path.abspath(json_file)}")


Upload your package document: .txt, .docx, or .pdf


Saving packages based on themes (1).pdf to packages based on themes (1) (10).pdf

================ DRAFT JSON OUTPUT ================

[
  {
    "name": "Everest Base Camp Trek for Beginners",
    "shortName": "EBC Trek for Beginners",
    "subcategories": [
      "expeditions-7000-8000m",
      "mountain-sunrise-escapes",
      "classic-short-treks"
    ],
    "trips": [
      "namche-bazaar-trek"
    ],
    "destinations": [
      "everest-region",
      "kathmandu"
    ],
    "country": "Nepal",
    "province": "Koshi Province",
    "district": "Solukhumbu",
    "cityOrVillage": "Khumbu / Lukla",
    "latitude": 27.9958,
    "longitude": 86.8284,
    "duration": 16,
    "description": "Cold rooms, long steps, thin air, repeated meals, basic toilets, and early mornings are part of Everest before Base Camp appears. This trek is built for first-time trekkers who want preparation before the trail teaches everything the hard way. You follow the classic EBC route with extra attention to p

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>